In [3]:
%pip install torch --break-system-packages
%pip install numpy --break-system-packages
%pip install pandas --break-system-packages





[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/PrathamShah/Library/Application Support/pipx/venvs/jupyterlab/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/PrathamShah/Library/Application Support/pipx/venvs/jupyterlab/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached pandas-3.0.5-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
Using cached pandas-3.0.5-cp314-cp314-macosx_11_0_arm64.whl (10.0 MB)

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/PrathamShah/Library/Application Support/pipx/venvs/jupyterlab/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install scikit-learn --break-system-packages

  Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.18.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl (8.2 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.24.0-py3-none-any.whl (461 kB)
Using cached scipy-1.18.0-cp314-cp314-macosx_14_0_arm64.whl (20.4 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn]0m 4/5 [scikit-learn]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/PrathamShah/Library/Application Support/pipx/venvs/jupyterlab/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use u

In [5]:
%pip install matplotlib --break-system-packages

  Using cached matplotlib-3.11.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (80 kB)
  Using cached contourpy-1.3.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp314-cp314-macosx_10_15_universal2.whl.metadata (118 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.1 kB)
  Using cached pillow-12.3.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (9.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.11.1-cp314-cp314-macosx_11_0_arm64.whl (9.3 MB)
Using cached contourpy-1.3.3-cp314-cp314-macosx_11_0_arm64.whl (273 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.63.0-cp314-cp314-macosx_10_15_universal2.whl (2.9 MB)
Using cached kiwisolver-1.5.0-cp314-cp314-macosx_11_0_arm64.whl (64 kB)
Using cached pillow-12.3.0-cp314-cp314-macosx_11_0_arm64.whl (4.8 MB)
Using cached pyparsing-3.3.2

In [8]:
VERSION = 3
 
import os
import json
import time
import numpy as np
import pandas as pd
 
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
 
from sklearn.model_selection import train_test_split, GroupKFold, GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from scipy.stats import pearsonr, spearmanr
 
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
 
import warnings
warnings.filterwarnings('ignore')
 
DATA_PATH = os.environ.get(
    "SUPPRESSION_CSV",
    r"/Users/PrathamShah/Desktop/Tubingen /Codes/Final_Dataset/community_index_final.csv"
)
OUTDIR = os.environ.get("/Users/PrathamShah/Desktop/Tubingen /Codes/MLP", ".")
os.makedirs(OUTDIR, exist_ok=True)
 
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GSS_SPLITS = int(os.environ.get("GSS_SPLITS", 10))   # 20 in the tree notebooks; NN fits cost more
MAX_EPOCHS = int(os.environ.get("MAX_EPOCHS", 200))
 
print(f"Device: {DEVICE}   torch {torch.__version__}")
 
 
def set_seed(seed=SEED):

    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
 
 
set_seed()
 
 
df = pd.read_csv(DATA_PATH)
 
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nTarget (community_suppression) statistics:")
print(df['community_suppression'].describe().round(4))
print("\nRows per diversity level:")
print(df['diversity_level'].value_counts().sort_index())
print("\nRows per community size:")
print(df['community_size'].value_counts().sort_index())
 
 
r0_cols = [f"R0_c{k}" for k in range(1, 21)]
active_cols = [f"active_c{k}" for k in range(1, 21)]
 
if VERSION == 1:
    species_cols = [f"Species_{i:04d}" for i in range(2, 102)]
    feature_cols = species_cols + r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Full (species + nutrients + size)"
elif VERSION == 2:
    feature_cols = r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Nutrients + size (no species)"
elif VERSION == 3:
    feature_cols = r0_cols + active_cols
    version_label = "Pure nutrients only (no species, no size)"
else:
    raise ValueError("VERSION must be 1, 2, or 3")
 
extra_cols = [c for c in feature_cols if c not in r0_cols + active_cols]
 
print(f"\nVersion: {version_label}")
print(f"Features: {len(feature_cols)}")
 
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"{len(missing)} columns not found. First 3: {missing[:3]}")
 
X = df[feature_cols]
y = df['community_suppression']
groups = df["broth_id"]
print(f"X: {X.shape}   y range: [{y.min():.3f}, {y.max():.3f}]")
 
strat_key = df["community_size"].astype(str) + "_" + df["diversity_level"].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key)
print(f"\nTrain: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}")
 
groups_train = df.loc[X_train.index, "broth_id"]
 
 
def inner_holdout(X_tr, y_tr, groups_tr, frac=0.15, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    fit_idx, val_idx = next(gss.split(X_tr, y_tr, groups=groups_tr))
    return (X_tr.iloc[fit_idx], y_tr.iloc[fit_idx],
            X_tr.iloc[val_idx], y_tr.iloc[val_idx])
 

Device: cpu   torch 2.13.0
Dataset shape: 75,000 rows x 168 columns

Target (community_suppression) statistics:
count    75000.0000
mean         0.5166
std          0.3328
min          0.0000
25%          0.2331
50%          0.4961
75%          0.8426
max          0.9995
Name: community_suppression, dtype: float64

Rows per diversity level:
diversity_level
1     15000
2     15000
4     15000
8     15000
16    15000
Name: count, dtype: int64

Rows per community size:
community_size
12    15000
16    15000
24    15000
32    15000
48    15000
Name: count, dtype: int64

Version: Pure nutrients only (no species, no size)
Features: 40
X: (75000, 40)   y range: [0.000, 0.999]

Train: 60,000   Test: 15,000


In [9]:
# mlp model definitions

class SuppressionMLP(nn.Module):
    """Feed-forward network mapping 40 nutrient features to one suppression value."""
 
    def __init__(self, n_features, hidden=(256, 128, 64), dropout=0.15,
                 use_batchnorm=True, output_activation='linear'):
        super().__init__()
 
        n_features = int(n_features)
        hidden = tuple(int(h) for h in hidden)

        self.n_features = n_features
        self.hidden = hidden
 
        layers = []
        prev = n_features
        for h in hidden:
            layers.append(nn.Linear(prev, h))

            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))

            layers.append(nn.GELU())

 
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            prev = h
 
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(prev, 1)
        self.output_activation = output_activation
 
    def forward(self, x):
        if x.dim() != 2 or x.shape[1] != self.n_features:
            raise ValueError(
                f"SuppressionMLP expected input of shape [batch, {self.n_features}], "
                f"got {tuple(x.shape)}. If the second number is wrong you are "
                f"feeding a different feature set than the model was built for; "
                f"if there are three dimensions the input was not flattened.")

        z = self.head(self.body(x)).squeeze(-1)
        if self.output_activation == 'sigmoid':
            z = torch.sigmoid(z)
        return z

 

In [10]:
def make_loader(X_arr, y_arr, batch_size=512, shuffle=False):
    """Wrap numpy arrays into a torch DataLoader.
    """
    ds = TensorDataset(torch.tensor(X_arr, dtype=torch.float32),
                       torch.tensor(y_arr, dtype=torch.float32))
 
    drop_last = False
    if shuffle and len(X_arr) % batch_size == 1 and len(X_arr) > batch_size:
        drop_last = True

 
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)
 
 
@torch.no_grad()
def predict(model, X_arr, batch_size=4096, clip=(0.0, 1.0)):
    """Run the model in eval mode and return a flat numpy array.
 

    """
    model.eval()
    out = []
    for i in range(0, len(X_arr), batch_size):
        xb = torch.tensor(X_arr[i:i + batch_size], dtype=torch.float32, device=DEVICE)
        out.append(model(xb).cpu().numpy())
    pred = np.concatenate(out)
    return np.clip(pred, *clip) if clip is not None else pred
 
 
ARCH_DEFAULTS = dict(hidden=(256, 128, 64), dropout=0.15,
                     use_batchnorm=True, output_activation='linear')
 
 
def resolve_arch(params, n_features):

    arch = {k: params.get(k, v) for k, v in ARCH_DEFAULTS.items()}
    arch['hidden'] = tuple(int(h) for h in arch['hidden'])
    arch['n_features'] = int(n_features)
    return arch
 
 
def train_mlp(X_fit, y_fit, X_val, y_val, params, max_epochs=MAX_EPOCHS,
              patience=20, seed=SEED, verbose=False, record=False):
    """Fit one MLP with early stopping on a group-aware validation set.
 
    The validation set here always comes from inner_holdout, meaning whole
    broths held out of the training broths. The test set never appears, which is
    the same leakage discipline we applied to the tree models after the
    early-stopping fix.
    """
    set_seed(seed)
 
    model = SuppressionMLP(
        n_features=X_fit.shape[1],
        hidden=params.get('hidden', (256, 128, 64)),
        dropout=params.get('dropout', 0.15),
        use_batchnorm=params.get('use_batchnorm', True),
        output_activation=params.get('output_activation', 'linear'),
    ).to(DEVICE)
 
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params.get('lr', 1e-3),
        weight_decay=params.get('weight_decay', 1e-4))

 
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=max(3, patience // 3))

    criterion = nn.MSELoss()

 
    train_loader = make_loader(X_fit, y_fit,
                               batch_size=params.get('batch_size', 512), shuffle=True)
 
    best_val, best_state, best_epoch, bad = np.inf, None, 0, 0
    hist = {'train': [], 'valid': []}
 
    for epoch in range(1, max_epochs + 1):
        model.train()
        running, n_seen = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
 
            optimizer.zero_grad(set_to_none=True)

 
            loss = criterion(model(xb), yb)
            loss.backward()

 
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

 
            optimizer.step()
 
            running += loss.item() * len(xb)
            n_seen += len(xb)
 
        tr_loss = running / n_seen
        val_pred = predict(model, X_val, clip=None)
        val_loss = float(np.mean((val_pred - y_val) ** 2))
        hist['train'].append(tr_loss)
        hist['valid'].append(val_loss)
        scheduler.step(val_loss)
 
        if val_loss < best_val - 1e-6:
            best_val, best_epoch, bad = val_loss, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        else:
            bad += 1
            if bad >= patience:
                break
 
        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"    epoch {epoch:>3}  train MSE={tr_loss:.5f}  val MSE={val_loss:.5f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")
 
    model.load_state_dict(best_state)   # restore the best epoch, not the last
    return (model, best_epoch, best_val, hist) if record else (model, best_epoch, best_val)
 
 

In [11]:
# initial model 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

 
y_train_np = y_train.values.astype(np.float32)
y_test_np = y_test.values.astype(np.float32)
 

Xi_fit, yi_fit, Xi_val, yi_val = inner_holdout(X_train, y_train, groups_train)
si = StandardScaler().fit(Xi_fit)
m_init, ep_init, val_init = train_mlp(
    si.transform(Xi_fit).astype(np.float32), yi_fit.values.astype(np.float32),
    si.transform(Xi_val).astype(np.float32), yi_val.values.astype(np.float32),
    params={}, verbose=True)
yp_init = predict(m_init, si.transform(X_test).astype(np.float32))
print(f"\nMLP (initial):  R2={r2_score(y_test, yp_init):.4f}  "
      f"MAE={mean_absolute_error(y_test, yp_init):.4f}  best_epoch={ep_init}")

    epoch   1  train MSE=0.07351  val MSE=0.06093  lr=1.00e-03
    epoch  10  train MSE=0.03460  val MSE=0.04431  lr=1.00e-03
    epoch  20  train MSE=0.03356  val MSE=0.04372  lr=5.00e-04
    epoch  30  train MSE=0.03340  val MSE=0.04384  lr=2.50e-04

MLP (initial):  R2=0.6958  MAE=0.1330  best_epoch=12


In [17]:
# tuning

RUN_TUNING = True
 

# Wider grid for RUN_TUNING = True:
param_grid = [
     
     dict(hidden=(128, 64),           dropout=0.10, lr=1e-3, weight_decay=1e-4, batch_size=512),
     dict(hidden=(256, 128, 64),      dropout=0.15, lr=1e-3, weight_decay=1e-4, batch_size=512),
     dict(hidden=(512, 256, 128),     dropout=0.20, lr=5e-4, weight_decay=1e-3, batch_size=512),
     dict(hidden=(512, 512, 256, 128),dropout=0.30, lr=5e-4, weight_decay=1e-3, batch_size=1024),
     dict(hidden=(256, 128, 64),      dropout=0.00, lr=1e-3, weight_decay=0.0,  batch_size=512),
 ]
 
 
def grid_search_mlp(X_tr, y_tr, groups_tr, grid, n_splits=3):
    """Group-aware search scored on held-out-broth MSE."""
    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    print(f"Searching {len(grid)} candidates x {n_splits} group folds")
    for i, params in enumerate(grid, start=1):
        fold_mse = []
        for tr_idx, va_idx in gkf.split(X_tr, y_tr, groups=groups_tr):
            Xa, ya = X_tr.iloc[tr_idx], y_tr.iloc[tr_idx]
            Xb, yb = X_tr.iloc[va_idx], y_tr.iloc[va_idx]
            sc = StandardScaler().fit(Xa)
            # inner split of the training fold, so early stopping never sees Xb
            ga = groups_tr.iloc[tr_idx]
            Xa1, ya1, Xa2, ya2 = inner_holdout(Xa, ya, ga, frac=0.15, seed=SEED + i)
            m, _, _ = train_mlp(sc.transform(Xa1).astype(np.float32),
                                ya1.values.astype(np.float32),
                                sc.transform(Xa2).astype(np.float32),
                                ya2.values.astype(np.float32),
                                params, max_epochs=min(MAX_EPOCHS, 80), patience=12)
            pb = predict(m, sc.transform(Xb).astype(np.float32))
            fold_mse.append(mean_squared_error(yb, pb))
        rows.append({'candidate': i - 1,
                     **{k: str(v) for k, v in params.items()},
                     'mean_mse': float(np.mean(fold_mse)),
                     'std_mse': float(np.std(fold_mse))})
        print(f"  [{i}/{len(grid)}] {params} -> MSE {np.mean(fold_mse):.5f}")
    return pd.DataFrame(rows).sort_values('mean_mse').reset_index(drop=True)
    # The 'candidate' column is what maps a row of this table back to its entry
    # in param_grid. Without it, reset_index(drop=True) throws that mapping
    # away and the winner cannot be recovered.
 
 
if RUN_TUNING:
    tune_df = grid_search_mlp(X_train, y_train, groups_train, param_grid)
    tune_df.to_csv(os.path.join(OUTDIR, "mlp_tuning.csv"), index=False)
    best_params = param_grid[int(tune_df.loc[0, 'candidate'])]
    print(f"\nBest params by held-out-broth MSE: {best_params}")
else:
    best_params = param_grid[0]
    print(f"Using params (tuning skipped): {best_params}")
 
 

Searching 5 candidates x 3 group folds
  [1/5] {'hidden': (128, 64), 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 512} -> MSE 0.09469
  [2/5] {'hidden': (256, 128, 64), 'dropout': 0.15, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 512} -> MSE 0.07347
  [3/5] {'hidden': (512, 256, 128), 'dropout': 0.2, 'lr': 0.0005, 'weight_decay': 0.001, 'batch_size': 512} -> MSE 0.07890
  [4/5] {'hidden': (512, 512, 256, 128), 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.001, 'batch_size': 1024} -> MSE 0.12179
  [5/5] {'hidden': (256, 128, 64), 'dropout': 0.0, 'lr': 0.001, 'weight_decay': 0.0, 'batch_size': 512} -> MSE 0.08947

Best params by held-out-broth MSE: {'hidden': (256, 128, 64), 'dropout': 0.15, 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 512}


In [19]:
# final model
Xf_fit, yf_fit, Xf_val, yf_val = inner_holdout(X_train, y_train, groups_train)
scaler_final = StandardScaler().fit(Xf_fit)
 
t0 = time.time()
model_final, best_epoch, best_val, hist = train_mlp(
    scaler_final.transform(Xf_fit).astype(np.float32), yf_fit.values.astype(np.float32),
    scaler_final.transform(Xf_val).astype(np.float32), yf_val.values.astype(np.float32),
    best_params, verbose=True, record=True)
print(f"Training time: {time.time() - t0:.1f}s   best epoch: {best_epoch}")
 
fig_tc, ax_tc = plt.subplots(figsize=(9, 4))
ax_tc.plot(range(1, len(hist['train']) + 1), hist['train'], color='steelblue', lw=1.5,
           label='Train MSE')
ax_tc.plot(range(1, len(hist['valid']) + 1), hist['valid'], color='darkorange', lw=1.5,
           label='Validation MSE')
ax_tc.axvline(best_epoch, color='red', linestyle='--', lw=1.2, label=f'Best epoch = {best_epoch}')
ax_tc.set_xlabel('Epoch')
ax_tc.set_ylabel('MSE')
ax_tc.set_title(f'MLP Training Curve -- best epoch={best_epoch}, val MSE={best_val:.5f}')
ax_tc.legend(fontsize=9)
ax_tc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'mlp_training_curve.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: mlp_training_curve.png")
 
X_test_sf = scaler_final.transform(X_test).astype(np.float32)
yp_final = predict(model_final, X_test_sf)
 
r2_final = r2_score(y_test, yp_final)
mae_final = mean_absolute_error(y_test, yp_final)
rmse_final = float(np.sqrt(mean_squared_error(y_test, yp_final)))
n_params = sum(p.numel() for p in model_final.parameters())
 
print(f"\nMLP final:  R2={r2_final:.4f}  MAE={mae_final:.4f}  RMSE={rmse_final:.4f}")
print(f"Parameters: {n_params:,}")
 

    epoch   1  train MSE=0.07351  val MSE=0.06093  lr=1.00e-03
    epoch  10  train MSE=0.03460  val MSE=0.04431  lr=1.00e-03
    epoch  20  train MSE=0.03356  val MSE=0.04372  lr=5.00e-04
    epoch  30  train MSE=0.03340  val MSE=0.04384  lr=2.50e-04
Training time: 7.3s   best epoch: 12
Saved: mlp_training_curve.png

MLP final:  R2=0.6958  MAE=0.1330  RMSE=0.1833
Parameters: 52,609


In [20]:
# gkf

gkf = GroupKFold(n_splits=5)
fold_rows = []
oof_pred = np.full(len(X), np.nan)
 
print("\nGroupKFold evaluation (held-out broths):")
 
for fold_num, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]
    g_tr = groups.iloc[tr_idx]
    assert len(set(g_tr).intersection(set(groups.iloc[te_idx]))) == 0
 
    sc = StandardScaler().fit(X_tr)
    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(X_tr, y_tr, g_tr, frac=0.15,
                                                   seed=42 + fold_num)
    m_fold, ep_fold, _ = train_mlp(
        sc.transform(Xg_fit).astype(np.float32), yg_fit.values.astype(np.float32),
        sc.transform(Xg_val).astype(np.float32), yg_val.values.astype(np.float32),
        best_params, seed=SEED + fold_num)
 
    yp_fold = predict(m_fold, sc.transform(X_te).astype(np.float32))
    oof_pred[te_idx] = yp_fold
 
    row = {'fold': fold_num, 'n_test': len(X_te),
           'r2': r2_score(y_te, yp_fold),
           'mae': mean_absolute_error(y_te, yp_fold),
           'rmse': float(np.sqrt(mean_squared_error(y_te, yp_fold))),
           'pearson': pearsonr(y_te, yp_fold).statistic,
           'spearman': spearmanr(y_te, yp_fold).statistic,
           'epochs': ep_fold}
    fold_rows.append(row)
    print(f"  Fold {fold_num}: R2={row['r2']:.4f}  MAE={row['mae']:.4f}  "
          f"RMSE={row['rmse']:.4f}  Pearson={row['pearson']:.4f}  "
          f"Spearman={row['spearman']:.4f}  epochs={row['epochs']}")
 
folds_df = pd.DataFrame(fold_rows)
 
r2_oof = r2_score(y, oof_pred)
mae_oof = mean_absolute_error(y, oof_pred)
rmse_oof = float(np.sqrt(mean_squared_error(y, oof_pred)))
 
print(f"\nPOOLED out-of-fold over all {len(X):,} rows:")
print(f"  R2={r2_oof:.4f}  MAE={mae_oof:.4f}  RMSE={rmse_oof:.4f}  "
      f"Spearman={spearmanr(y, oof_pred).statistic:.4f}")
print(f"\nInterpolation R2 = {r2_final:.4f}   Held-out-broth R2 = {r2_oof:.4f}   "
      f"Gap = {r2_final - r2_oof:.4f}")
 


GroupKFold evaluation (held-out broths):
  Fold 1: R2=0.5325  MAE=0.1739  RMSE=0.2248  Pearson=0.7310  Spearman=0.7379  epochs=4
  Fold 2: R2=0.3071  MAE=0.2037  RMSE=0.2728  Pearson=0.5664  Spearman=0.5496  epochs=45
  Fold 3: R2=0.4511  MAE=0.1921  RMSE=0.2559  Pearson=0.6881  Spearman=0.6785  epochs=2
  Fold 4: R2=0.4008  MAE=0.1951  RMSE=0.2620  Pearson=0.6580  Spearman=0.6552  epochs=13
  Fold 5: R2=0.5167  MAE=0.1755  RMSE=0.2233  Pearson=0.7371  Spearman=0.7279  epochs=29

POOLED out-of-fold over all 75,000 rows:
  R2=0.4421  MAE=0.1881  RMSE=0.2486  Spearman=0.6640

Interpolation R2 = 0.6958   Held-out-broth R2 = 0.4421   Gap = 0.2537


In [21]:
# groupshufflesplit
gss = GroupShuffleSplit(n_splits=GSS_SPLITS, test_size=0.20, random_state=42)
gss_rows = []
 
for split_num, (tr_idx, te_idx) in enumerate(gss.split(X, y, groups=groups), start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]
    g_tr = groups.iloc[tr_idx]
    assert len(set(g_tr).intersection(set(groups.iloc[te_idx]))) == 0
 
    sc = StandardScaler().fit(X_tr)
    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(X_tr, y_tr, g_tr, frac=0.15,
                                                   seed=42 + split_num)
    m, ep, _ = train_mlp(
        sc.transform(Xg_fit).astype(np.float32), yg_fit.values.astype(np.float32),
        sc.transform(Xg_val).astype(np.float32), yg_val.values.astype(np.float32),
        best_params, seed=SEED + split_num)
    yp = predict(m, sc.transform(X_te).astype(np.float32))
 
    gss_rows.append({'split': split_num,
                     'r2': r2_score(y_te, yp),
                     'mae': mean_absolute_error(y_te, yp),
                     'rmse': float(np.sqrt(mean_squared_error(y_te, yp))),
                     'epochs': ep})
    print(f"Split {split_num:02d}: R2={gss_rows[-1]['r2']:.4f}  "
          f"MAE={gss_rows[-1]['mae']:.4f}  RMSE={gss_rows[-1]['rmse']:.4f}  epochs={ep}")
 
gss_df = pd.DataFrame(gss_rows)
print(f"\nGroupShuffleSplit held-out-broth summary ({GSS_SPLITS} splits):")
for c in ['r2', 'mae', 'rmse']:
    print(f"  {c:<6} mean={gss_df[c].mean():.4f}  std={gss_df[c].std():.4f}")
 
 

Split 01: R2=0.2119  MAE=0.2527  RMSE=0.3106  epochs=5
Split 02: R2=0.5179  MAE=0.1802  RMSE=0.2347  epochs=25
Split 03: R2=0.3326  MAE=0.2081  RMSE=0.2604  epochs=39
Split 04: R2=0.5681  MAE=0.1645  RMSE=0.2099  epochs=48
Split 05: R2=0.4942  MAE=0.1753  RMSE=0.2202  epochs=54
Split 06: R2=0.4992  MAE=0.1754  RMSE=0.2256  epochs=92
Split 07: R2=0.2774  MAE=0.2278  RMSE=0.2784  epochs=3
Split 08: R2=0.4115  MAE=0.1862  RMSE=0.2505  epochs=2
Split 09: R2=0.5002  MAE=0.1748  RMSE=0.2278  epochs=43
Split 10: R2=0.5714  MAE=0.1742  RMSE=0.2164  epochs=47

GroupShuffleSplit held-out-broth summary (10 splits):
  r2     mean=0.4384  std=0.1250
  mae    mean=0.1919  std=0.0285
  rmse   mean=0.2434  std=0.0318


In [22]:
# per condition breakdown
print("Performance by diversity level and community size")
 
results_df = pd.DataFrame({
    'y_true': y_test.values,
    'y_pred': yp_final,
    'residual': y_test.values - yp_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size': df.loc[X_test.index, 'community_size'].values,
})
 
 
def condition_metrics(g):
    y_t, y_p = g['y_true'].values, g['y_pred'].values
    return {'n': len(y_t), 'r2': r2_score(y_t, y_p),
            'mae': mean_absolute_error(y_t, y_p),
            'rmse': float(np.sqrt(mean_squared_error(y_t, y_p))),
            'pearson': pearsonr(y_t, y_p).statistic,
            'spearman': spearmanr(y_t, y_p).statistic,
            'bias': float(np.mean(y_p - y_t)), 'std_true': float(np.std(y_t))}
 
 
hdr = (f"{'':6} {'R2':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} "
       f"{'Bias':>8} {'std(y)':>7} {'N':>6}")
 
print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m = condition_metrics(grp)
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson']:>8.4f} {m['spearman']:>9.4f} {m['bias']:>8.4f} "
          f"{m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R2 is unreliable here. Read MAE and Spearman instead.")
 
print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp)
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson']:>8.4f} {m['spearman']:>9.4f} {m['bias']:>8.4f} "
          f"{m['std_true']:>7.4f} {m['n']:>6,}")
 
 

Performance by diversity level and community size

By diversity level:
            R2     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
D=1     0.8447  0.0912  0.1706   0.9195    0.9526   0.0020  0.4330  3,000   
D=2     0.6669  0.1474  0.2057   0.8176    0.8188   0.0095  0.3564  3,000   
D=4     0.4513  0.1750  0.2181   0.6748    0.6860  -0.0083  0.2944  3,000   
D=8     0.5590  0.1388  0.1727   0.7481    0.7174  -0.0018  0.2600  3,000   
D=16    0.3513  0.1127  0.1387   0.5958    0.5646  -0.0098  0.1722  3,000   
  * std(y) < 0.08: R2 is unreliable here. Read MAE and Spearman instead.

By community size:
            R2     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
N=12    0.4879  0.1688  0.2343   0.7993    0.7819   0.1262  0.3274  3,000
N=16    0.6613  0.1357  0.1926   0.8383    0.8299   0.0673  0.3309  3,000
N=24   

In [24]:
# figures

output_name = os.path.join(OUTDIR, f"mlp_v{VERSION}_results.png")
 
fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(f'MLP -- {version_label}\n'
             f'R2={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}  '
             f'held-out-broth R2={r2_oof:.3f}  params={n_params:,}',
             fontsize=12, fontweight='bold', y=0.99)
 
divs = sorted(results_df['diversity_level'].unique())
sizes = sorted(results_df['community_size'].unique())
 
ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1, 2, 4, 8, 16], ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8', '#984ea3']):
    mask = results_df['diversity_level'].values == d
    ax1.scatter(y_test.values[mask], yp_final[mask], alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0, 1], [0, 1], 'r--', lw=1.5)
ax1.set_xlabel('True suppression')
ax1.set_ylabel('Predicted')
ax1.set_title(f'Predicted vs Actual  R2={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)
 
ax2 = fig.add_subplot(2, 3, 2)
resid = results_df['residual'].values
for n, col in zip([12, 16, 24, 32, 48], ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e']):
    mask = results_df['community_size'].values == n
    ax2.scatter(yp_final[mask], resid[mask], alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5, color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Residual')
ax2.set_title('Residuals by Community Size')
ax2.legend(fontsize=7, markerscale=3)
 
ax3 = fig.add_subplot(2, 3, 3)
 
 
def feat_col(f):
    if 'Species' in f:
        return '#2196F3'
    if 'R0' in f:
        return '#4CAF50'
    if 'active' in f:
        return '#FF9800'
    return '#9C27B0'
 

 
ax4 = fig.add_subplot(2, 3, 4)
r2_div = [r2_score(results_df[results_df['diversity_level'] == d]['y_true'],
                   results_df[results_df['diversity_level'] == d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R2={r2_final:.3f}')
ax4.set_xlabel('Diversity level')
ax4.set_ylabel('R2')
ax4.set_title('R2 by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)
 
ax5 = fig.add_subplot(2, 3, 5)
r2_sz = [r2_score(results_df[results_df['community_size'] == n]['y_true'],
                  results_df[results_df['community_size'] == n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R2={r2_final:.3f}')
ax5.set_xlabel('Community size N')
ax5.set_ylabel('R2')
ax5.set_title('R2 by Community Size')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)
 
# Panel 6: the diagnostic that matters most for a neural network on this data
ax6 = fig.add_subplot(2, 3, 6)
ax6.scatter(y.values, oof_pred, alpha=0.10, s=2, c='darkorange', label='Held-out broths (OOF)')
ax6.scatter(y_test.values, yp_final, alpha=0.10, s=2, c='steelblue', label='Interpolation test')
ax6.plot([0, 1], [0, 1], 'r--', lw=1.5)
ax6.set_xlabel('True suppression')
ax6.set_ylabel('Predicted')
ax6.set_title(f'Interpolation vs Held-out Broth\n'
              f'R2 {r2_final:.3f} -> {r2_oof:.3f}  (gap {r2_final - r2_oof:.3f})')
ax6.legend(fontsize=8, markerscale=4)
 
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(output_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nPlot saved: {output_name}")
 
 


Plot saved: ./mlp_v3_results.png


In [25]:
# save model
model_path = os.path.join(OUTDIR, f"mlp_v{VERSION}_model.pt")
torch.save({
    'state_dict': model_final.state_dict(),
    'arch': {**resolve_arch(best_params, len(feature_cols)),
             'n_features': model_final.n_features,
             'hidden': list(model_final.hidden)},
    # n_features and hidden are read back OFF THE TRAINED MODEL rather than
    # re-derived from best_params. Anything re-derived can drift from what was
    # actually built; what the model reports about itself cannot.
    'scaler_mean': scaler_final.mean_.tolist(),
    'scaler_scale': scaler_final.scale_.tolist(),
    'feature_cols': feature_cols,
}, model_path)
# The scaler is saved INSIDE the checkpoint on purpose. A network without its
# scaler is useless, and keeping them in separate files is the most common way
# to end up applying the wrong normalisation months later.
 
pd.DataFrame({
    'y_true': y_test.values, 'y_pred': yp_final,
    'residual': y_test.values - yp_final,
    'diversity_level': results_df['diversity_level'].values,
    'community_size': results_df['community_size'].values,
}).to_csv(os.path.join(OUTDIR, f"mlp_v{VERSION}_predictions.csv"), index=False)
 
pd.DataFrame({
    'y_true': y.values, 'oof_pred': oof_pred, 'broth_id': df['broth_id'].values,
    'diversity_level': df['diversity_level'].values,
    'community_size': df['community_size'].values,
}).to_csv(os.path.join(OUTDIR, f"mlp_v{VERSION}_oof_heldout_broth.csv"), index=False)
 
folds_df.to_csv(os.path.join(OUTDIR, f"mlp_v{VERSION}_groupkfold.csv"), index=False)
gss_df.to_csv(os.path.join(OUTDIR, f"mlp_v{VERSION}_groupshufflesplit.csv"), index=False)

 
with open(os.path.join(OUTDIR, f"mlp_v{VERSION}_summary.json"), 'w') as f:
    json.dump({'model': 'mlp', 'version': VERSION, 'version_label': version_label,
               'n_features': len(feature_cols), 'n_params': int(n_params),
               'best_epoch': int(best_epoch),
               'params': {k: str(v) for k, v in best_params.items()},
               'r2_interpolation': float(r2_final), 'mae_interpolation': float(mae_final),
               'rmse_interpolation': float(rmse_final),
               'r2_heldout_broth': float(r2_oof), 'mae_heldout_broth': float(mae_oof),
               'rmse_heldout_broth': float(rmse_oof),
               'gss_r2_mean': float(gss_df['r2'].mean()),
               'gss_r2_std': float(gss_df['r2'].std())}, f, indent=2)
 
print(f"Model saved: {model_path}")
 

Model saved: ./mlp_v3_model.pt


In [26]:
ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
m2 = SuppressionMLP(**ckpt['arch']).to(DEVICE)
m2.load_state_dict(ckpt['state_dict'])
sc_mean = np.array(ckpt['scaler_mean'], dtype=np.float32)
sc_scale = np.array(ckpt['scaler_scale'], dtype=np.float32)
X_check = ((X_test.values[:5] - sc_mean) / sc_scale).astype(np.float32)
assert np.allclose(predict(m2, X_check), yp_final[:5], atol=1e-5)
print("Reloaded model reproduces predictions exactly.")
 

Reloaded model reproduces predictions exactly.


In [27]:
# summary

print(f"\n{'=' * 68}")
print(f"SUMMARY -- MLP VERSION {VERSION}: {version_label}")
print(f"{'=' * 68}")
print(f"Features:                    {len(feature_cols)}")
print(f"Parameters:                  {n_params:,}")
print(f"Best epoch:                  {best_epoch}")
print(f"Interpolation R2:            {r2_final:.4f}")
print(f"Interpolation MAE:           {mae_final:.4f}")
print(f"Interpolation RMSE:          {rmse_final:.4f}")
print(f"Held-out-broth R2 (pooled):  {r2_oof:.4f}")
print(f"Held-out-broth MAE:          {mae_oof:.4f}")
print(f"Generalization gap (R2):     {r2_final - r2_oof:.4f}")
print(f"GroupShuffleSplit R2:        {gss_df['r2'].mean():.4f} +/- {gss_df['r2'].std():.4f}")



SUMMARY -- MLP VERSION 3: Pure nutrients only (no species, no size)
Features:                    40
Parameters:                  52,609
Best epoch:                  12
Interpolation R2:            0.6958
Interpolation MAE:           0.1330
Interpolation RMSE:          0.1833
Held-out-broth R2 (pooled):  0.4421
Held-out-broth MAE:          0.1881
Generalization gap (R2):     0.2537
GroupShuffleSplit R2:        0.4384 +/- 0.1250
